# Fluorescence decay and lifetime analysis

Every decay fit in tttrlib is reached the same way: **name the model, hand it the
measurement, read the answer back**. There is one interface rather than one per
estimator, and what a model's parameters, setup values and results *mean* comes
from the registry rather than from a per-model convention you have to memorise.

The narrative guide is [Fluorescence Decay Fitting](../fit-guide.rst); the plotted
walk-through is the `plot_decay_fit_interface` gallery example.


## The four pieces

| Piece | What it is |
|---|---|
| `DecayFit2` | The model, built by registry name. Immutable, so build once and reuse. |
| `DecayFitProblem` | The measurement: data, IRF, background, reference patterns. |
| `DecayFitConstraints` | What the optimiser may move: free, fixed, or linked. |
| the outcome | `parameters`, `results`, `objective`. The fitted curve stays on `problem.model`. |


## What can I fit?

The registry lists the models and describes each one, so nothing here has to be
remembered or kept in sync by hand.


In [ ]:
import numpy as np
import tttrlib

tttrlib.fit_names()


In [ ]:
# Everything a caller needs in order to *offer* fit23 to a user.
entry = tttrlib.registry('fit')['fit23']
print(entry['label'])
print(entry['summary'])
for name, prop in entry['params_schema']['properties'].items():
    unit = prop.get('unit', '')
    held = ' (held by default)' if prop.get('fixed_default') else ''
    print(f"  {name:<8} default {prop['default']:<6} {unit}{held}")


## A synthetic measurement

A narrow instrument response, given twice — parallel then perpendicular in one
array, the layout the polarisation-resolved models expect.

One setup value deserves attention: the **excitation period bounds the lifetime
search**, because the decay is convolved over one period. A lifetime longer than
the period cannot be measured.


In [ ]:
n_bins, dt, period, true_tau = 128, 0.032, 32.0, 2.0

channel = np.arange(n_bins)
irf_half = np.exp(-0.5 * ((channel - 10) / 1.2) ** 2)
irf_half /= irf_half.sum()
irf = np.concatenate([irf_half, irf_half])
background = np.zeros(2 * n_bins)

# Built from the registry's documented defaults; only what we name is applied.
setup = tttrlib.setup_vector('fit23', dt=dt, period=period,
                             soft_bifl_scatter_flag=False)
dict(zip(tttrlib.decay_fit_setup_names('fit23'), setup))


## Build the fit and the problem


In [ ]:
fit = tttrlib.DecayFit2('fit23', setup, irf.tolist())

problem = tttrlib.DecayFitProblem(2, n_bins, dt)   # 2 detection channels
problem.irf = tttrlib.VectorDouble(irf.tolist())
problem.background = tttrlib.VectorDouble(background.tolist())

print('parameters:', list(tttrlib.decay_fit_parameter_names('fit23')))
print('results   :', list(tttrlib.decay_fit_result_names('fit23')))


## Simulate a decay

`model_curve` gives the curve the model predicts, **independent of any data** —
which is what simulating needs. Its sibling `evaluate` scores parameters *against*
data and scales the curve to the observed counts, so it returns zeros before any
data exist.


In [ ]:
curve = np.asarray(fit.model_curve([true_tau, 0.0, 0.0, 1.0], problem))
expected = curve / curve.sum() * 30000
data = np.random.default_rng(1).poisson(expected).astype(float)
problem.data = tttrlib.VectorDouble(data.tolist())

print(f'{data.sum():.0f} photons over {2 * n_bins} channels')


## Fit

The link vector says what may move: `0` free, `-1` held, and any positive value
ties every slot carrying it to one shared value. `default_links` takes the
registry's advice — scatter and anisotropy are barely identifiable from a short
decay, so they are held unless a measured IRF and background make them meaningful.


In [ ]:
constraints = tttrlib.DecayFitConstraints(
    tttrlib.VectorInt32(tttrlib.default_links('fit23')))

outcome = fit.fit([0.5, 0.0, 0.0, 1.0], constraints, problem)  # poor start on purpose
print(f'true      {true_tau:.3f} ns')
print(f'recovered {outcome.parameters[0]:.3f} ns')


## Read the answer by name

`twoIstar` compares the model against a hypothetical perfectly-fitting one, so a
good fit to counting data sits near 1.

> **`converged` is not `correct`.** It reports that the optimiser met its
> tolerance. A lifetime longer than the excitation period converges onto that
> bound and still reports `True` — if a fitted lifetime sits exactly on the
> period, lengthen the period rather than believe the number.


In [ ]:
tttrlib.results_as_dict('fit23', list(outcome.results))


## Swapping the model is a change of name

Nothing else about the call moves. This is the point of the interface: previously
each estimator was a different class with a different parameter layout and a
different result convention.

`fit24` needs a background with some weight in it — its likelihood carries a
background term that is undefined against an all-zero pattern, in which case the
fit returns NaN without moving.


In [ ]:
problem24 = tttrlib.DecayFitProblem(2, n_bins, dt)
problem24.irf = tttrlib.VectorDouble(irf.tolist())
problem24.background = tttrlib.VectorDouble((np.ones(2 * n_bins) / (2 * n_bins)).tolist())
problem24.data = tttrlib.VectorDouble(data.tolist())

fit24 = tttrlib.DecayFit2('fit24', tttrlib.setup_vector('fit24', dt=dt, period=period),
                          irf.tolist())
out24 = fit24.fit([1.0, 0.0, 3.0, 0.5, 0.0],
                  tttrlib.DecayFitConstraints(tttrlib.VectorInt32([0, -1, 0, 0, -1])),
                  problem24)
print(f'tau1 {out24.parameters[0]:.2f} ns, tau2 {out24.parameters[2]:.2f} ns, 2I* {out24.objective:.2f}')


A bi-exponential model fitted to single-exponential data is not identifiable — the
two lifetimes trade against each other. The *call* is the same; the answer is only
meaningful if the photophysics calls for two components.


## Fitting many decays

Batching belongs to the interface rather than to each model, so it works for every
fit. Rows are spread across workers with the GIL released; the model is shared
because it is immutable, and only per-row state is copied. This is the path a
burst analysis or a FLIM image takes.


In [ ]:
n_rows = 200
matrix = np.random.default_rng(7).poisson(np.tile(expected, (n_rows, 1))).astype(float)

batch = fit.fit_many(problem, matrix.ravel().tolist(), n_rows, 2 * n_bins,
                     [0.5, 0.0, 0.0, 1.0], constraints)
taus = np.asarray(batch.parameters).reshape(n_rows, 4)[:, 0]
print(f'tau = {taus.mean():.3f} +/- {taus.std():.3f} ns over {n_rows} decays')


The spread across repeats is the statistical uncertainty of the estimator at this
photon count — which is what to quote, rather than an error estimate from a single
fit.


## Bounds are priors

A hard box is the degenerate case of a prior: uniform inside, impossible outside.
So a parameter needs one concept, not two. Priors serialise as JSON in the same
form ChiSurf uses, so they cross between the two losslessly.


In [ ]:
c = tttrlib.DecayFitConstraints(tttrlib.VectorInt32([0, -1, -1, -1]))
c.set_prior_json(0, '{"kind": "lognormal", "mu": 0.7, "sigma": 0.3}')

out = fit.fit([0.5, 0.0, 0.0, 1.0], c, problem)
print(f'with a lognormal prior on tau: {out.parameters[0]:.3f} ns')


## Convolving: the recursion or the transform?

A model decay is compared against data only after it has been convolved with the
instrument response, and there are two ways to do that. The received wisdom —
"convolution is a multiplication in frequency space, so use an FFT" — is sound
for convolving a response with an arbitrary signal, and **wrong** for a decay
made of exponentials.

Both paths cost `O(n_bins x n_rates)`. The recursion does one multiply-add per
rate and bin; the closed-form periodic spectrum does one complex *division* per
rate and frequency. The transform is not what dominates — evaluating the closed
form is — so the frequency domain buys no better scaling, only worse constants,
and the recursion is SIMD-optimised on top of that.

Start by checking the two give the same answer.

In [ ]:
import numpy as np
import tttrlib

RECURSIVE, SPECTRAL = 0, 1

n_bins = 1024
rates, weights = [1.0 / 50.0, 1.0 / 12.0], [0.7, 0.3]
i = np.arange(n_bins)
irf = np.exp(-0.5 * ((i - 100.0) / 12.0) ** 2)

a = np.asarray(tttrlib.dfa_convolve(rates, weights, irf.tolist(), n_bins, 0.0, RECURSIVE))
b = np.asarray(tttrlib.dfa_convolve(rates, weights, irf.tolist(), n_bins, 0.0, SPECTRAL))

print("largest relative difference: %.2e" % (np.abs(a - b).max() / a.max()))

Machine precision — which took a correction rather than a tolerance.

The recursion applies the trapezoid rule to the convolution integral, and the
kernel that leaves is `exp(-k L)` at every lag except `L = 0`, where it leaves
one half. Halving one sample is subtracting half a delta, and a delta has a flat
spectrum, so the entire difference is a constant `1/2` subtracted from the
periodic spectrum.

Skip it and the two differ by `(1 + exp(-k)) / 2` — a factor that depends on the
**rate**, so it does not divide out of a rate spectrum but reweights it. In a
model built to measure the relative weights of a FRET-rate distribution, that is
an error in the answer, not in the last digit.

In [ ]:
for k in (0.005, 0.01, 0.05, 0.1, 0.3):
    print("k = %5.3f per bin -> uncorrected discrepancy would be %5.2f%%"
          % (k, 100.0 * (1.0 - (1.0 + np.exp(-k)) / 2.0)))

### Which is faster, and by how much

Measured by `benchmarks/bench_convolution.py` on an M1 Pro:

| bins | rates | recursion | spectral | spectral is |
|-----:|------:|----------:|---------:|:------------|
| 1024 | 1 | 28 us | 47 us | 1.7x slower |
| 1024 | 16 | 55 us | 253 us | 4.6x slower |
| 1024 | 64 | 147 us | 911 us | 6.2x slower |
| 4096 | 64 | 581 us | 3569 us | 6.1x slower |

The gap **widens** with the rate count — exactly the regime a
donor x FRET x anisotropy outer product lives in, so the frequency domain is at
its worst where a rate-spectrum model would most want it.

**Use the recursion.** It is the default, so that is also the "do nothing"
answer. Reach for the spectral path for the three things the recursion cannot
do: an arbitrary measured pattern (an autofluorescence reference is not a sum of
exponentials), an independent cross-check, and a response broad enough to wrap
around the period — the recursion starts at bin 0 as though nothing preceded it,
so it cannot see the wrapped part.

In [ ]:
broad = np.exp(-0.5 * ((i - 100.0) / 300.0) ** 2)
wide_a = np.asarray(tttrlib.dfa_convolve(rates, weights, broad.tolist(), n_bins, 0.0, RECURSIVE))
wide_b = np.asarray(tttrlib.dfa_convolve(rates, weights, broad.tolist(), n_bins, 0.0, SPECTRAL))

print("compact response: %.1e" % (np.abs(a - b).max() / a.max()))
print("broad (wrapping): %.1e" % (np.abs(wide_a - wide_b).max() / wide_a.max()))

### A sub-bin timeshift needs neither choice

A whole-bin shift is a roll; a fractional one is not, and rounding it to the
nearest bin biases the fitted lifetime when bins are coarse. Pass a fractional
`shift_bins` and the recursion borrows one transform of the *response*, at a cost
independent of the number of rates.

The shift goes on the response and never on the decay. A phase ramp is
band-limited interpolation, so it rings at a step — and the decay has one, where
the next pulse arrives. Shift the decay and its tail oscillates and goes
**negative**, which a Poisson likelihood cannot take the logarithm of. The
response is a compact pulse near zero at both ends, so it shifts cleanly.

In [ ]:
half = np.asarray(tttrlib.dfa_convolve(rates, weights, irf.tolist(), n_bins, 0.5, RECURSIVE))
print("model stays positive under a half-bin shift:", half.min() > 0)

decay = np.asarray(tttrlib.dfa_periodic_decay([0.1], [1.0], 64))
w = np.arange(decay.size // 2 + 1)
rung = np.fft.irfft(np.fft.rfft(decay) * np.exp(-2j * np.pi * w * 0.5 / 64), 64)
print("shifting the *decay* instead goes negative:", rung.min() < 0)

### The response must be sized for the channels you declared

A shared instrument response is only shared when there is *one* channel to share
it with. A polarisation-resolved model reads `n_channels * n_bins` samples
straight out of the array, so handing it a response sized for one channel is an
out-of-bounds read — not a shorthand. Both `fit` and `model_curve` refuse it.

In [ ]:
bad = tttrlib.DecayFitProblem(2, n_bins, dt)
bad.irf = tttrlib.VectorDouble(irf[:n_bins].tolist())          # one channel...
bad.background = tttrlib.VectorDouble(np.zeros(n_bins).tolist())
print("validation_error():", bad.validation_error())           # ...for a 2-channel problem

try:
    fit.model_curve([2.0, 0.0, 0.38, 1.2], bad)
except ValueError as exc:
    print("refused:", exc)

It is worth saying why this is an error rather than a tolerated shorthand: it
used to be accepted. The curve came back with plausible numbers, the read ran
past the end of the heap buffer, and the process died later in unrelated code —
which is a very expensive way to find out.

## Deprecated API

The former per-estimator classes (`Fit23`/`Fit24`/`Fit25`/`Fit26`, the `fit23`-style
helpers, `DecayFitData`) still work from Python, warn on use, and are **removed in
0.29**. They are rebuilt on top of the interface above and return the same numbers.
Port to `DecayFit2`: the layouts then come from the registry, and the same code
works for every model.
